In [9]:
# 지난 Week1 코드 재사용

import tensorflow as tf
import numpy as np
import pandas as pd

path_to_train = tf.keras.utils.get_file(
    "train.txt", "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"
)
path_to_test = tf.keras.utils.get_file(
    "test.txt", "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt"
)

train_text = open(path_to_train, "rb").read().decode(encoding="utf-8")
test_text = open(path_to_test, "rb").read().decode(encoding="utf-8")

# 정답 라벨 만들기
train_Y = np.array(
    [
        [int(row.split("\t")[2])]
        for row in train_text.split("\n")[1:]
        if row.count("\t") > 0
    ]
)

test_Y = np.array(
    [
        [int(row.split("\t")[2])]
        for row in test_text.split("\n")[1:]
        if row.count("\t") > 0
    ]
)

import re


def clean_str(string):
    string = re.sub(r"[^가-힣A-Za-z0-9(),!?\'\`]", " ", string)
    string = re.sub(r"\'s", " 's", string)
    string = re.sub(r"\'ve", " 've", string)
    string = re.sub(r"n\'t", " n't", string)
    string = re.sub(r"\'re", " 're", string)
    string = re.sub(r"\'d", " 'd", string)
    string = re.sub(r"\'ll", " 'll", string)
    string = re.sub(r",", " , ", string)
    string = re.sub(r"!", " ! ", string)
    string = re.sub(r"\(", " \( ", string)
    string = re.sub(r"\)", " \) ", string)
    string = re.sub(r"\?", " \? ", string)
    string = re.sub(r"\s{2,}", " ", string)
    string = re.sub(r"\'{2,}", "'", string)
    string = re.sub(r"\'", "", string)

    return string.lower()


train_text_X = [
    row.split("\t")[1] for row in train_text.split("\n")[1:] if row.count("\t") > 0
]
train_text_X = [clean_str(sentence) for sentence in train_text_X]
# 문장을 띄어쓰기 단위로 단어 분리
sentences = [sentence.split(" ") for sentence in train_text_X]

VOCAB_SIZE = 2000
MAX_LEN = 25

vectorize_layer = tf.keras.layers.TextVectorization(
    standardize="lower_and_strip_punctuation",
    split="whitespace",
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=MAX_LEN,
)
vectorize_layer.adapt(train_text_X)

train_X = vectorize_layer(train_text_X)

In [2]:
test_text_X = [
    row.split("\t")[1] for row in test_text.split("\n")[1:] if row.count("\t") > 0
]

test_X = vectorize_layer(test_text_X)

print("test_X Shape: ", test_X.shape)
print("test_Y Shape: ", test_Y.shape)

test_X Shape:  (50000, 25)
test_Y Shape:  (50000, 1)


In [3]:
# 모델 하이퍼파라미터 정의
VOCAB_SIZE = 2000
EMBEDDING_DIM = 128
MAX_LEN = 25
EPOCHS = 10
BATCH_SIZE = 32

In [7]:
model = tf.keras.Sequential(
    [
        tf.keras.layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_shape=(MAX_LEN,)),
        tf.keras.layers.GlobalAveragePooling1D(),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ]
)

In [8]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

In [10]:
history = model.fit(
    train_X,
    train_Y,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    verbose=1,
)

Epoch 1/10
3750/3750 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.7213 - loss: 0.5083 - val_accuracy: 0.7433 - val_loss: 0.4881
Epoch 2/10
3750/3750 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.7507 - loss: 0.4639 - val_accuracy: 0.7582 - val_loss: 0.4581
Epoch 3/10
3750/3750 ━━━━━━━━━━━━━━━━━━━━ 3s 918us/step - accuracy: 0.7538 - loss: 0.4564 - val_accuracy: 0.7593 - val_loss: 0.4556
Epoch 4/10
3750/3750 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.7555 - loss: 0.4523 - val_accuracy: 0.7549 - val_loss: 0.4582
Epoch 5/10
3750/3750 ━━━━━━━━━━━━━━━━━━━━ 3s 866us/step - accuracy: 0.7570 - loss: 0.4477 - val_accuracy: 0.7506 - val_loss: 0.4649
Epoch 6/10
3750/3750 ━━━━━━━━━━━━━━━━━━━━ 3s 889us/step - accuracy: 0.7579 - loss: 0.4450 - val_accuracy: 0.7559 - val_loss: 0.4654
Epoch 7/10
3750/3750 ━━━━━━━━━━━━━━━━━━━━ 4s 948us/step - accuracy: 0.7606 - loss: 0.4417 - val_accuracy: 0.7586 - val_loss: 0.4582
Epoch 8/10
3750/3750 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.7624 - loss: 0

In [11]:
test_loss, test_acc = model.evaluate(test_X, test_Y, verbose=0)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}")

Test Loss: 0.4822, Test Accuracy: 0.7384


In [12]:
example_sentences = [
    "이 영화 진짜 재미있어요",
    "완전 지루하고 별로였음",
    "배우 연기는 좋았지만 스토리가 아쉬웠다",
]

example_seq = vectorize_layer(example_sentences)
pred = model.predict(example_seq)

for s, p in zip(example_sentences, pred):
    print(f"문장: {s}")
    print(f"긍정 확률: {p[0]:.4f}")
    print(f"결과: ", "긍정 😊" if p[0] > 0.5 else "부정 😞")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
문장: 이 영화 진짜 재미있어요
긍정 확률: 0.9924
결과:  긍정 😊
문장: 완전 지루하고 별로였음
긍정 확률: 0.0249
결과:  부정 😞
문장: 배우 연기는 좋았지만 스토리가 아쉬웠다
긍정 확률: 0.0425
결과:  부정 😞


In [ ]:
# 실습 미션 코드를 작성해주세요.
# 1. Epoch=30 으로 늘려 모델 학습해주세요.
# 2. matplotlib를 이용하여 Epoch 10 과 Epoch 30을 비교하여 시각화해주세요.

import matplotlib.pyplot as plt

EPOCHS_LONG = 30

model_30 = tf.keras.Sequential(
    [
        tf.keras.layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_shape=(MAX_LEN,)),
        tf.keras.layers.GlobalAveragePooling1D(),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ]
)
model_30.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

history_30 = model_30.fit(
    train_X,
    train_Y,
    epochs=EPOCHS_LONG,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    verbose=1,
)

In [ ]:
# Epoch 10 vs Epoch 30 비교 시각화
import matplotlib.pyplot as plt

epochs_10 = range(1, len(history.history["loss"]) + 1)
epochs_30 = range(1, len(history_30.history["loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ---- Loss 비교 ----
ax = axes[0]
ax.plot(epochs_10, history.history["loss"],
        color="tab:blue", linestyle="-", label="Epoch 10 - train")
ax.plot(epochs_10, history.history["val_loss"],
        color="tab:blue", linestyle="--", label="Epoch 10 - val")
ax.plot(epochs_30, history_30.history["loss"],
        color="tab:red", linestyle="-", label="Epoch 30 - train")
ax.plot(epochs_30, history_30.history["val_loss"],
        color="tab:red", linestyle="--", label="Epoch 30 - val")
ax.axvline(x=10, color="gray", linestyle=":", alpha=0.6, label="Epoch 10 종료 시점")
ax.set_title("Loss 비교 (Epoch 10 vs Epoch 30)")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()
ax.grid(True, alpha=0.3)

# ---- Accuracy 비교 ----
ax = axes[1]
ax.plot(epochs_10, history.history["accuracy"],
        color="tab:blue", linestyle="-", label="Epoch 10 - train")
ax.plot(epochs_10, history.history["val_accuracy"],
        color="tab:blue", linestyle="--", label="Epoch 10 - val")
ax.plot(epochs_30, history_30.history["accuracy"],
        color="tab:red", linestyle="-", label="Epoch 30 - train")
ax.plot(epochs_30, history_30.history["val_accuracy"],
        color="tab:red", linestyle="--", label="Epoch 30 - val")
ax.axvline(x=10, color="gray", linestyle=":", alpha=0.6, label="Epoch 10 종료 시점")
ax.set_title("Accuracy 비교 (Epoch 10 vs Epoch 30)")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
